# 🐖 05 | Auditoría del modelo YOLO y conteo de instancias

**RECURSO: CPU — SIN ACELERADOR**. No vuelve a entrenar ni consultar el conjunto test con el modelo; examina únicamente resultados ya calculados, archivos y distribución de anotaciones.

**Proyecto:** Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖

Los notebooks son independientes: cada uno utiliza los productos guardados por la etapa anterior, sin repetir descarga o entrenamiento.


## Propósito

- Confirmar **cuántas imágenes e instancias anotadas** existen en el dataset y en las 373 imágenes de test.
- Separar **instancias repetidas** de animales únicos (no hay identificación individual).
- Analizar las 60 épocas y las métricas ya guardadas de YOLO11n-seg.
- Exportar un **Excel**, dos figuras, reporte JSON y ZIP de auditoría. Se usan los resultados previos del notebook 04; no se ejecuta de nuevo sobre test.


In [ ]:
%pip -q install pandas openpyxl pyyaml matplotlib pillow


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, os

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
if not ROOT.is_dir():
    raise FileNotFoundError('No encuentro la carpeta del proyecto. Comprueba que Colab monta la cuenta correcta y que el nombre no cambió: '+str(ROOT))
DATA=ROOT/'01_Datasets'/'FaroPigSeg'
EXTRACT=DATA/'extraido'/'FaroPigSeg'
SPLIT=DATA/'split_fecha_v1'
RES=ROOT/'03_Resultados'
EXP=ROOT/'05_Exportaciones_ZIP'
for s in ['01_Exploracion','02_Segmentacion','03_Metricas','04_Figuras','05_Reportes','06_Entrenamiento']:
    (RES/s).mkdir(parents=True,exist_ok=True)
EXP.mkdir(exist_ok=True,parents=True)
MANIFEST=SPLIT/'particion_fecha_manifest.json'
YAML=SPLIT/'data_fecha.yaml'
if not (MANIFEST.is_file() and YAML.is_file()):
    raise FileNotFoundError('Faltan los archivos de partición congelada; ejecutar notebook 02 en CPU antes de continuar.')
part=json.loads(MANIFEST.read_text(encoding='utf-8'))
assert len(set(part['asignaciones_fecha'].values())-{'train','val','test'})==0
print('Carpeta del proyecto:',ROOT)
print('Fechas por grupo:',{k:sum(v==k for v in part['asignaciones_fecha'].values()) for k in ('train','val','test')})


In [ ]:
import pandas as pd,numpy as np,matplotlib.pyplot as plt
from collections import Counter
from matplotlib.ticker import MaxNLocator
INDEX=RES/'01_Exploracion'/'FaroPigSeg_inventario.xlsx'
RESULTS_CSV=RES/'06_Entrenamiento'/'YOLO11n_fecha_v1'/'results.csv'
TEST_JSON=RES/'03_Metricas'/'CERDOS_04_Evaluacion_test.json'
if not INDEX.is_file():
    raise FileNotFoundError(f'Falta {INDEX}; conserva la auditoría original o ejecuta notebook 01 (CPU).')
if not RESULTS_CSV.is_file() or not TEST_JSON.is_file():
    raise FileNotFoundError('Faltan los resultados de YOLO 03/04. No se repetirá test automáticamente.')
imgs=pd.read_excel(INDEX,sheet_name='Imagenes')
need={'imagen','instancias_validas','particion'}
assert need.issubset(imgs.columns), f'Faltan columnas en inventario: {need-set(imgs.columns)}'
if imgs['imagen'].duplicated().any():
    raise RuntimeError('Nombres de archivo repetidos: no se puede enlazar con seguridad al inventario. Revisar origen.')
imgs['fecha']=imgs['imagen'].astype(str).str.extract(r'^(\d{4}-\d{2}-\d{2})')[0]
assert imgs['fecha'].notna().all(), 'Alguna imagen no tiene fecha en el nombre.'
imgs['nuevo_split']=imgs['fecha'].map(part['asignaciones_fecha'])
assert imgs['nuevo_split'].notna().all(), 'Partición congelada incompleta.'
for split in ('train','val','test'):
    listed=[Path(x.strip()).name for x in (SPLIT/f'{split}.txt').read_text(encoding='utf8').splitlines() if x.strip()]
    found=imgs.loc[imgs.nuevo_split.eq(split),'imagen'].tolist()
    assert sorted(listed)==sorted(found), f'Inconsistencia entre manifiesto e inventario en {split}'
resumen=(imgs.groupby('nuevo_split',as_index=False).agg(imagenes=('imagen','size'),instancias=('instancias_validas','sum'),fechas=('fecha','nunique')))
fechas=imgs.groupby(['fecha','nuevo_split'],as_index=False).agg(imagenes=('imagen','size'),instancias=('instancias_validas','sum'))
historico=pd.read_csv(RESULTS_CSV);historico.columns=historico.columns.str.strip()
yolo=json.loads(TEST_JSON.read_text(encoding='utf8'))
assert int(yolo['test_count'])==int(resumen.set_index('nuevo_split').loc['test','imagenes'])
print('TOTAL imágenes:',len(imgs),'TOTAL instancias anotadas:',int(imgs['instancias_validas'].sum()))
print(resumen.to_string(index=False))
print('Imágenes representadas visualmente en el ZIP 04: 12; NO es el tamaño del test.')
print('Épocas registradas:',len(historico))
print('YOLO test mAP50-95 de máscaras:',yolo['seg_map50_95'])


In [ ]:
from matplotlib import pyplot as plt
REPORT_DIR=RES/'05_Reportes';MET=RES/'03_Metricas';FIG=RES/'04_Figuras'
output=MET/'CERDOS_05_Auditoria_Resultados.xlsx'
with pd.ExcelWriter(output,engine='openpyxl') as w:
    resumen.to_excel(w,sheet_name='Resumen_particiones',index=False)
    fechas.to_excel(w,sheet_name='Fechas_y_conteos',index=False)
    imgs[['imagen','particion','fecha','nuevo_split','instancias_validas']].to_excel(w,sheet_name='Imagenes',index=False)
    historico.to_excel(w,sheet_name='Epocas_YOLO',index=False)
    pd.DataFrame([yolo]).to_excel(w,sheet_name='Metricas_test_YOLO',index=False)

fig,ax=plt.subplots(figsize=(11,5))
f=fechas.sort_values('fecha')
ax.bar(range(len(f)),f['instancias'])
ax.set_xticks(range(len(f)));ax.set_xticklabels(f['fecha'],rotation=70,ha='right',fontsize=8)
ax.set_ylabel('Instancias anotadas (no individuos únicos)');ax.set_title('FaroPigSeg — apariciones de cerdos por fecha')
fig.tight_layout();figure_counts=FIG/'CERDOS_05_Anotaciones_por_Fecha.png';fig.savefig(figure_counts,dpi=170);plt.show()

cols=['train/seg_loss','val/seg_loss','metrics/mAP50-95(M)']
fig,axes=plt.subplots(1,2,figsize=(11,4))
for col,label in [('train/seg_loss','Pérdida de segmentación (train)'),('val/seg_loss','Pérdida de segmentación (val)')]:
    if col in historico:axes[0].plot(historico['epoch'],historico[col],label=label)
axes[0].set_title('Historial YOLO');axes[0].set_xlabel('Época');axes[0].legend()
if 'metrics/mAP50-95(M)' in historico:
    axes[1].plot(historico['epoch'],historico['metrics/mAP50-95(M)'])
axes[1].set_title('mAP50-95 máscara (validación)');axes[1].set_xlabel('Época');axes[1].set_ylim(0,1)
fig.tight_layout();figure_curves=FIG/'CERDOS_05_Curvas_YOLO.png';fig.savefig(figure_curves,dpi=170);plt.show()

report={'etapa':'05_auditoria_CPU','fecha_utc':datetime.now(timezone.utc).isoformat(),
'num_imagenes':int(len(imgs)),'num_instancias_anotadas':int(imgs.instancias_validas.sum()),
'num_imagenes_test':int(resumen.set_index('nuevo_split').loc['test','imagenes']),
'num_instancias_test':int(resumen.set_index('nuevo_split').loc['test','instancias']),
'numero_fotografias_ejemplo_previas':12,'epocas_yolo':int(len(historico)),
'metricas_yolo_test':yolo,'nota':'Instancias anotadas no equivalen a cerdos físicos únicos. No se reevalúa test.'}
jsonpath=REPORT_DIR/'CERDOS_05_Auditoria.json'
jsonpath.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf8')
print('Excel de auditoría:',output)


In [ ]:
import zipfile
files_out=[output,figure_counts,figure_curves,jsonpath]
export=EXP/'CERDOS_05_Auditoria_Resultados.zip'
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in files_out:z.write(p,arcname=p.name)
print('✅ ZIP:',export)
from google.colab import files as colab_files
colab_files.download(str(export))


**Interpretación:** se cuentan apariciones anotadas; sin identificadores únicos ni secuencias no se puede inferir cuántos cerdos físicos hubo. Continuar con **06 Mask R-CNN (GPU)**.
